In [12]:
import os, json, time, queue, threading, contextlib, httpx, sounddevice as sd
from dotenv import load_dotenv
from deepgram import DeepgramClient
from deepgram.speak.v1.types import SpeakV1Text

load_dotenv()
dg  = DeepgramClient(api_key=os.environ["DEEPGRAM_API_KEY"])
LLM = httpx.Client(base_url="https://api.cerebras.ai/v1", timeout=30,
                   headers={"Authorization": f"Bearer {os.environ['CEREBUS']}"})

MODEL = "gpt-oss-120b"
VOICE, SPEED = "aura-2-jupiter-en", 1.2
KEYTERMS = ["Kubernetes", "Priya", "Postgres", "standup"]

# "Respond shortly" on its own still returns markdown lists and "≈238,900 km",
# which the voice reads out literally. The rest keeps it speakable.
SYSTEM = ("Respond shortly. You are a voice assistant and your reply is read aloud: "
          "answer in one or two plain spoken sentences. Never use markdown, lists, "
          "bullets, asterisks, headings, or emoji. Write numbers and units as words.")

RATE = int(sd.query_devices(kind="input")["default_samplerate"])
OUT = sd.RawOutputStream(samplerate=24000, channels=1, dtype="int16", latency="low")
OUT.start()

_tail, FLUSHES, SPEAKING = b"", 0, False

def play(chunk):
    global _tail                       # chunk boundaries don't respect 16-bit frames
    buf = _tail + chunk
    cut = len(buf) - len(buf) % 2
    try:
        OUT.write(buf[:cut])
    except Exception:
        pass                           # device swapped mid-utterance; keep the pump alive
    _tail = buf[cut:]

def use_output(match=None):
    """Move playback to another device. An open stream never follows the system
    default, and PortAudio snapshots the device list when it loads - so AirPods
    paired after the kernel started are invisible until this re-scans."""
    global OUT
    try:
        OUT.abort(); OUT.close()
    except Exception:
        pass
    sd._terminate(); sd._initialize()
    dev = None
    if match:
        dev = next(i for i, d in enumerate(sd.query_devices())
                   if match.lower() in d["name"].lower() and d["max_output_channels"] > 0)
    OUT = sd.RawOutputStream(samplerate=24000, channels=1, dtype="int16",
                             latency="low", device=dev)
    OUT.start()
    print("output ->", sd.query_devices(OUT.device)["name"])

# One socket held open for the whole session: 0.22s to first audio instead of 0.84s
# reconnecting per line.
SPEAKER = contextlib.ExitStack()
SOCK = SPEAKER.enter_context(
    dg.speak.v1.connect(model=VOICE, encoding="linear16", sample_rate=24000, speed=SPEED))

def _pump():
    global FLUSHES
    try:
        for m in SOCK:
            if isinstance(m, bytes):
                play(m)                # blocking, so by the time Flushed lands below,
            elif getattr(m, "type", None) == "Flushed":
                FLUSHES += 1           # every chunk before it is already out the speaker
    except Exception:
        pass                           # socket closed
threading.Thread(target=_pump, daemon=True).start()

history = [{"role": "system", "content": SYSTEM}]

def reply(heard):
    """Pipe tokens straight from Cerebras into the open TTS socket, flushing at each
    sentence end so it starts talking on sentence one, not on the whole answer."""
    global SPEAKING
    history.append({"role": "user", "content": heard})
    SPEAKING = True
    before, flushes, answer = FLUSHES, 0, ""
    with LLM.stream("POST", "/chat/completions", json={
            "model": MODEL, "messages": history, "stream": True,
            # gpt-oss reasons before answering and those tokens come out of the same
            # budget - at effort "high" they can eat it entirely and return nothing.
            "reasoning_effort": "low", "max_completion_tokens": 400,
            "temperature": 0.6}) as r:
        if r.status_code != 200:
            print(f"  [cerebras {r.status_code}] {r.read().decode()[:160]}")  # 429 = per-minute cap
            SPEAKING = False
            return
        for line in r.iter_lines():
            if not line.startswith("data: "):
                continue
            body = line[6:]
            if body == "[DONE]":
                break
            tok = json.loads(body)["choices"][0]["delta"].get("content") or ""
            if not tok:
                continue
            answer += tok
            SOCK.send_text(SpeakV1Text(type="Speak", text=tok))
            if tok.rstrip()[-1:] in ".!?":
                SOCK.send_flush(); flushes += 1
    SOCK.send_flush(); flushes += 1                  # trailing text, plus a done marker
    history.append({"role": "assistant", "content": answer})
    print(f"  jarvis: {answer.strip() or '[empty - reasoning used the whole budget]'}")

    deadline = time.time() + 30                      # wait for the audio to actually finish
    while FLUSHES - before < flushes and time.time() < deadline:
        time.sleep(0.02)
    time.sleep(0.25)                                 # device buffer tail
    SPEAKING = False

print("ready")

||PaMacCore (AUHAL)|| Warning on line 521: err=''!obj'', msg=Unknown Error
||PaMacCore (AUHAL)|| Warning on line 441: err=''!obj'', msg=Unknown Error
||PaMacCore (AUHAL)|| Error on line 1332: err='-10851', msg=Audio Unit: Invalid Property Value


PortAudioError: Error opening RawOutputStream: Internal PortAudio error [PaErrorCode -9986]

In [ ]:
use_output("AirPods")

output -> Marcus’s AirPods


In [10]:
q = queue.Queue()
with dg.listen.v2.connect(
    model="flux-general-en", encoding="linear16", sample_rate=RATE,
    eager_eot_threshold=0.3, eot_threshold=0.5, eot_timeout_ms=2000,
    keyterm=KEYTERMS,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda d, *_: q.put(bytes(len(d)) if SPEAKING else bytes(d))):
        print("say something - interrupt the cell to hang up\n")   # mic feeds silence while
        try:                                                        # it talks, or it hears itself
            for m in sock:
                if getattr(m, "event", None) == "EndOfTurn" and m.transcript.strip():
                    print(f"you:    {m.transcript}")
                    reply(m.transcript)
        except KeyboardInterrupt:
            q.put(None)

say something - interrupt the cell to hang up

you:    Hello?
  jarvis: Hello! How can I help you today?
you:    Can you briefly summarize what the University of Auckland does?
  jarvis: The University of Auckland is a public research university in New Zealand that offers undergraduate and postgraduate programs across a wide range of disciplines, conducts cutting‑edge research, and engages with the community and industry.
you:    Okay.
  jarvis: Glad to help—let me know if you’d like more details on any specific area.
you:    What is your knowledge
  jarvis: I’m trained on a broad range of information up through June 2024, covering topics like science, history, technology, culture, and everyday knowledge.
you:    When?
  jarvis: My training data includes information available up to June 2024.
you:    what year did World War two end?
  jarvis: World War Two ended in nineteen forty‑five.
you:    Can you give me the exact date
  jarvis: The war ended on the second of September, nineteen f